# Assignment Text summariser powered by LLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
!pip install evaluate
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


## Checking prerequisites

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

CUDA available: True
Device count: 1
Current device: 0
Device name: Tesla T4
PyTorch version: 2.11.0+cu128


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import torch
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)

import transformers
print("Transformers version:", transformers.__version__)

Torch version: 2.11.0+cu128
Torch location: /usr/local/lib/python3.12/dist-packages/torch/__init__.py
Transformers version: 5.13.1


### Loading Dataset

In [ ]:
# VS code version
#dataset = load_from_disk("S:\Projects\Datasets\TextS\samsum_dataset")

# Colab version
dataset = load_from_disk("/content/drive/MyDrive/samsum_dataset")

#### Neural networks cannot process raw text directly. Text must first be converted into numerical representations. Modern LLMs achieve this using subword tokenization.

#### LLM's like gpt rely on tokenization methods like BPE, but for this dataset and T5 llm we will use SentencePiece

#### Lets take a look at token embedding and trannsformer, Text to text transfer transformer, Flan-T5-base (trained on 250M parameters), big enough to be called LLM

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Lets inspect the model

In [ ]:
model.config

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {


In [ ]:
model.shared

Embedding(32128, 768)

In [ ]:
model.encoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseGatedActDense(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
            (wo): Linear(in_features=2048, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
      

In [ ]:
model.decoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=F

In [ ]:
model.lm_head

Linear(in_features=768, out_features=32128, bias=False)

#### There is 1 difference in encoder and decoder as seen above, encoder dosent have cross attention layer. The encoder only reads the input sentence, and dosent need another sequence to attend to.

#### The decoder has already generated summary, but it also takes input from enncoder along with generated input which is cross verified while generating the optimal output. This process is also called self supervising learning.

### Also the tokenizer

In [ ]:
tokenizer.vocab_size

32100

In [ ]:
tokenizer.model_max_length

512

In [ ]:
tokenizer.pad_token

'<pad>'

In [ ]:
tokenizer.eos_token

'</s>'

In [ ]:
sample = dataset["train"][0]["dialogue"]
print(sample)

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)


In [ ]:
tokens = tokenizer.tokenize(sample)
print(tokens[:50])

['▁Amanda', ':', '▁I', '▁baked', '▁cookies', '.', '▁Do', '▁you', '▁want', '▁some', '?', '▁Jerry', ':', '▁Sure', '!', '▁Amanda', ':', '▁I', "'", 'll', '▁bring', '▁you', '▁tomorrow', '▁', ':', '-', ')']


In [ ]:
ids = tokenizer.encode(sample)
print(ids[:50])

[21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1]


### model already has pretrtained embedding lets extract one

In [ ]:
embeddings = model.get_input_embeddings().weight
print(embeddings.shape)

torch.Size([32128, 768])


In [ ]:
tokens = tokenizer.tokenize("hello")
print(tokens)

token = tokens[0]

token_id = tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

['▁hello']
▁hello
21820


#### The pretrainied embedding has 21820 as hello, i will make more sense when compared with surounding vectors and it will have closer semantic meaning

### T5 base dont need a positional embedding, its has Relative position, biased inside the attention mechanism

## Now lets take a look at attention block

In [ ]:
print(model.encoder.block[0].layer[0])

T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
    (relative_attention_bias): Embedding(32, 12)
  )
  (layer_norm): T5LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [ ]:
attn = model.encoder.block[0].layer[0].SelfAttention
print(attn)

T5Attention(
  (q): Linear(in_features=768, out_features=768, bias=False)
  (k): Linear(in_features=768, out_features=768, bias=False)
  (v): Linear(in_features=768, out_features=768, bias=False)
  (o): Linear(in_features=768, out_features=768, bias=False)
  (relative_attention_bias): Embedding(32, 12)
)


In [ ]:
print(attn.has_relative_attention_bias)

True


##### This shows that the model has relative attention type, rather than absolute that we see in classic gpt

In [ ]:
print(attn.relative_attention_bias)

Embedding(32, 12)


#### Here 32 is the relative distance buckets with 12 attention heads, model learns distance rather than positions, which is a smarter approach

In [ ]:
# Attention weights
attn.relative_attention_bias.weight

Parameter containing:
tensor([[ 3.3072e+00, -1.4124e+01,  2.2363e+00, -7.5515e+00,  8.4037e+00,
          5.4025e+00,  4.9113e-01,  2.5243e-01,  4.3401e+00,  6.6022e+00,
         -8.6801e+00, -2.5473e+01],
        [-2.5756e+01,  1.0481e+01,  8.4726e+00,  3.9471e+00,  9.8540e+00,
          1.7485e+00,  9.1644e+00,  6.1179e+00,  7.9472e+00, -4.2284e+00,
          2.8060e+00,  7.6758e+00],
        [-1.5956e+01,  8.7715e+00,  5.2965e+00,  4.5750e+00,  7.7746e+00,
          9.5001e-01,  8.6429e+00,  6.6384e+00,  7.5241e+00, -1.7510e+01,
          3.7001e+00,  8.0501e+00],
        [-1.5508e+01,  7.6623e+00,  4.6198e+00,  4.7793e+00,  6.7673e+00,
          1.9559e+00,  8.1014e+00,  6.7059e+00,  7.0760e+00, -1.9015e+01,
          3.9715e+00,  8.0325e+00],
        [-1.3945e+01,  7.0022e+00,  4.4271e+00,  4.7923e+00,  5.8716e+00,
          2.2086e+00,  7.6472e+00,  6.8344e+00,  6.7654e+00, -2.1642e+01,
          4.1278e+00,  7.9277e+00],
        [-1.5966e+01,  6.4181e+00,  4.3777e+00,  4.9517e+0

#### These are pretrained weights

# PreProcesssing

In [ ]:
# Dataset in use
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [ ]:
# Look at sample
dataset["train"][0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [ ]:
sample = dataset["train"][0]

prompt = f"""Summarize the following conversation.

Dialogue:
{sample['dialogue']}

Summary:"""

print(prompt)

Summarize the following conversation.

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:


### Applying the above example to whole dataset, because for FLAN an instruction prompt will give out better results

In [ ]:
# always load fresh
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 64

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def create_prompt(dialogue):
    return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

In [ ]:
def preprocess_function(batch):

    prompts = [
        create_prompt(dialogue)
        for dialogue in batch["dialogue"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset"
)

Tokenizing dataset:   0%|          | 0/818 [00:00<?, ? examples/s]

In [ ]:
# Check a sample
tokenized_dataset["train"][0]

{'input_ids': [12198,
  1635,
  1737,
  8,
  826,
  3634,
  5,
  5267,
  10384,
  10,
  21542,
  10,
  27,
  13635,
  5081,
  5,
  531,
  25,
  241,
  128,
  58,
  16637,
  10,
  10625,
  55,
  21542,
  10,
  27,
  31,
  195,
  830,
  25,
  5721,
  3,
  10,
  18,
  61,
  20698,
  10,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]}

In [ ]:
sample = tokenized_dataset["train"][0]

print(sample.keys())

print(sample["input_ids"][:20])
print(sample["labels"][:20])

print(tokenizer.decode(sample["input_ids"]))
print(tokenizer.decode(
    [x for x in sample["labels"] if x != -100]
))

dict_keys(['input_ids', 'attention_mask', 'labels'])
[12198, 1635, 1737, 8, 826, 3634, 5, 5267, 10384, 10, 21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128]
[21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]
Summarize the following conversation. Dialogue: Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-) Summary:</s>
Amanda baked cookies and will bring Jerry some tomorrow.</s>


## Fine tuning the data and model

In [ ]:
# data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

In [ ]:
# Get dataloader
train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)

#### The collator creates the decoder_input_ids automatically,, this process is called "teacher forcing", at every step the decoder is shown the correct previous word, not its own prediction that can be wrong.

In [ ]:
# Configuration
EPOCHS = 3
LEARNING_RATE = 3e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)


# Scheduler
num_training_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=int(0.05 * num_training_steps),
    num_training_steps=num_training_steps
)

In [ ]:
# Training loop
model.train()

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):

    total_loss = 0

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        scheduler.step()

        optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    print(total_loss / len(train_loader))

  0%|          | 0/7366 [00:00<?, ?it/s]

1.4283064976354802


  0%|          | 0/7366 [00:00<?, ?it/s]

1.3160706618955011


  0%|          | 0/7366 [00:00<?, ?it/s]

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/flan_t5_samsumV1"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
import os

SAVE_PATH = "/content/drive/MyDrive/flan_t5_samsumV1"

print(os.path.exists(SAVE_PATH))
print(os.listdir(SAVE_PATH) if os.path.exists(SAVE_PATH) else "Folder not found")

True
['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


### Testing the model on unseen data

In [ ]:
dialogue = """
Mira: Okay, we need to rethink the entire hypothesis. The data isn’t matching our model.
Leon: I’ve been saying that the entropy term is too weak. It collapses under noise.
Arjun: Or maybe the noise isn’t noise. Maybe it’s a pattern we haven’t recognised yet.
Mira: A hidden variable?
Leon: Possibly. But that means rewriting half the thesis.
Arjun: Better rewrite than defend something broken.
Mira: True. Let’s start with the core assumption: the system is stable.
Leon: Except it clearly isn’t.
Arjun: Unless stability is local, not global.
Mira: That… actually fits the anomalies.
Leon: Local stability pockets. Like micro‑equilibria.
Arjun: Yes! And the transitions between pockets create the spikes we saw.
Mira: Then our model needs a dynamic boundary function.
Leon: I can code that, but it’ll take hours.
Arjun: I’ll help. You handle the math, I’ll handle the implementation.
Mira: And I’ll rewrite the theoretical section to match the new structure.
Leon: Wait, what about the energy decay curve?
Arjun: It becomes nonlinear. That’s why the old model failed.
Mira: So we justify it using the new transition theory.
Leon: This is starting to make sense.
Arjun: Don’t jinx it.
Mira: Too late, he already did.
Leon: Fine, fine. Let’s keep going. What about the boundary collapse events?
Arjun: They’re rare, but predictable with the new function.
Mira: Then we can include them as edge cases in the appendix.
Leon: Good. Reviewers love edge cases.
Arjun: Reviewers love tearing them apart.
Mira: Which is why ours need to be airtight.
Leon: I’ll run simulations tonight.
Arjun: I’ll stay and help. We need this done.
Mira: And I’ll finish the revised thesis outline before midnight.
Leon: This is the closest we’ve been to a working theory.
Arjun: Let’s not waste the momentum.
Mira: Agreed. Back to work.
"""

In [ ]:
inputs = tokenizer(create_prompt(dialogue), return_tensors="pt").to(device)

summary_ids = model.generate(**inputs, max_new_tokens=64)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


Mira and Arjun are trying to figure out the model. Leon and Arjun will help them. Mira will finish the revised thesis outline.
